<a href="https://colab.research.google.com/github/maruwagner893-droid/cloud-provider-analytics/blob/main/notebooks/02_pipeline_etapa2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
!pip -q install pyspark

In [22]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("cloud-provider-analytics")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
print("Versión de Spark:", spark.version)

Versión de Spark: 4.0.5


In [23]:
import os

REPO = "/content/cloud-provider-analytics"

if not os.path.isdir(REPO):
    !git clone https://github.com/maruwagner893-droid/cloud-provider-analytics {REPO}

!unzip -q -o {REPO}/data/cloud_provider_challenge_dataset_v1.zip -d {REPO}/data/extracted

In [24]:
LANDING = os.path.join(REPO, "data/extracted/datalake/landing")
print("Landing:", LANDING)
print(sorted(os.listdir(LANDING)))

Landing: /content/cloud-provider-analytics/data/extracted/datalake/landing
['billing_monthly.csv', 'customers_orgs.csv', 'marketing_touches.csv', 'nps_surveys.csv', 'resources.csv', 'support_tickets.csv', 'usage_events_stream', 'users.csv']


In [25]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, BooleanType, DateType, DoubleType
)

schema_customers = StructType([
    StructField("org_id",           StringType(),  True),
    StructField("org_name",         StringType(),  True),
    StructField("industry",         StringType(),  True),
    StructField("hq_region",        StringType(),  True),
    StructField("plan_tier",        StringType(),  True),
    StructField("is_enterprise",    BooleanType(), True),
    StructField("signup_date",      DateType(),    True),
    StructField("sales_rep",        StringType(),  True),
    StructField("lifecycle_stage",  StringType(),  True),
    StructField("marketing_source", StringType(),  True),
    StructField("nps_score",        DoubleType(),  True),
])

customers = (
    spark.read
    .option("header", True)
    .schema(schema_customers)
    .csv(os.path.join(LANDING, "customers_orgs.csv"))
)

customers.printSchema()
customers.show(5, truncate=False)

# Verificación
print("Filas:", customers.count())                                                 # 80
print("signup_date nulos:", customers.filter(F.col("signup_date").isNull()).count())  # 0
print("nps_score nulos:", customers.filter(F.col("nps_score").isNull()).count())      # 11

root
 |-- org_id: string (nullable = true)
 |-- org_name: string (nullable = true)
 |-- industry: string (nullable = true)
 |-- hq_region: string (nullable = true)
 |-- plan_tier: string (nullable = true)
 |-- is_enterprise: boolean (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- sales_rep: string (nullable = true)
 |-- lifecycle_stage: string (nullable = true)
 |-- marketing_source: string (nullable = true)
 |-- nps_score: double (nullable = true)

+------------+---------------+-------------+----------+---------+-------------+-----------+---------+---------------+----------------+---------+
|org_id      |org_name       |industry     |hq_region |plan_tier|is_enterprise|signup_date|sales_rep|lifecycle_stage|marketing_source|nps_score|
+------------+---------------+-------------+----------+---------+-------------+-----------+---------+---------------+----------------+---------+
|org_xaji0y6d|Nimbus Labs 0  |Education    |sa-east   |standard |true         |2025-05-26 |rep_

In [26]:
BRONZE = os.path.join(REPO, "data/bronze")

customers_bronze = (
    customers
    .withColumn("ingest_ts", F.current_timestamp())
    .withColumn("source_file", F.col("_metadata.file_name"))
)

(
    customers_bronze
    .coalesce(1)                      # 80 filas: un solo archivo, sin archivos chicos
    .write.mode("overwrite")          # re-ejecutable: pisa el resultado anterior
    .partitionBy("plan_tier")
    .parquet(os.path.join(BRONZE, "customers_orgs"))
)

# Verificación: leer lo que se guardó
check = spark.read.parquet(os.path.join(BRONZE, "customers_orgs"))
check.printSchema()
check.select("org_id", "ingest_ts", "source_file").show(3, truncate=False)
print("Filas en Bronze:", check.count())   # 80

!ls -la {BRONZE}/customers_orgs

root
 |-- org_id: string (nullable = true)
 |-- org_name: string (nullable = true)
 |-- industry: string (nullable = true)
 |-- hq_region: string (nullable = true)
 |-- is_enterprise: boolean (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- sales_rep: string (nullable = true)
 |-- lifecycle_stage: string (nullable = true)
 |-- marketing_source: string (nullable = true)
 |-- nps_score: double (nullable = true)
 |-- ingest_ts: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- plan_tier: string (nullable = true)

+------------+--------------------------+------------------+
|org_id      |ingest_ts                 |source_file       |
+------------+--------------------------+------------------+
|org_xaji0y6d|2026-11-11 16:24:15.129788|customers_orgs.csv|
|org_pbhsahxt|2026-11-11 16:24:15.129788|customers_orgs.csv|
|org_8mdd4v30|2026-11-11 16:24:15.129788|customers_orgs.csv|
+------------+--------------------------+------------------+
only showing top

In [27]:
schema_users = StructType([
    StructField("user_id",    StringType(),  True),
    StructField("org_id",     StringType(),  True),
    StructField("email",      StringType(),  True),
    StructField("role",       StringType(),  True),
    StructField("active",     BooleanType(), True),
    StructField("created_at", DateType(),    True),
    StructField("last_login", DateType(),    True),
])

schema_billing = StructType([
    StructField("invoice_id",           StringType(), True),
    StructField("org_id",               StringType(), True),
    StructField("month",                DateType(),   True),
    StructField("subtotal",             DoubleType(), True),
    StructField("credits",              DoubleType(), True),
    StructField("taxes",                DoubleType(), True),
    StructField("currency",             StringType(), True),
    StructField("exchange_rate_to_usd", DoubleType(), True),
])

def ingest_master(name, schema, key, part_col):
    """Landing (CSV) -> Bronze (Parquet): tipifica, deduplica por clave y agrega metadatos."""
    df = (
        spark.read.option("header", True).schema(schema)
        .csv(os.path.join(LANDING, f"{name}.csv"))
        .withColumn("ingest_ts", F.current_timestamp())
        .withColumn("source_file", F.col("_metadata.file_name"))
    )
    n_in = df.count()
    df = df.dropDuplicates([key])
    n_out = df.count()
    out = os.path.join(BRONZE, name)
    df.coalesce(1).write.mode("overwrite").partitionBy(part_col).parquet(out)
    print(f"{name}: leídas={n_in}  duplicadas quitadas={n_in - n_out}  guardadas={n_out}  -> {out}")

ingest_master("customers_orgs",  schema_customers, "org_id",     "plan_tier")
ingest_master("users",           schema_users,     "user_id",    "role")
ingest_master("billing_monthly", schema_billing,   "invoice_id", "month")
# Verificación
users_b   = spark.read.parquet(os.path.join(BRONZE, "users"))
billing_b = spark.read.parquet(os.path.join(BRONZE, "billing_monthly"))
print("users last_login nulos:", users_b.filter(F.col("last_login").isNull()).count())      # 139
print("billing credits nulos:", billing_b.filter(F.col("credits").isNull()).count())        # 137
print("billing subtotal < 0:", billing_b.filter(F.col("subtotal") < 0).count())             # 13

customers_orgs: leídas=80  duplicadas quitadas=0  guardadas=80  -> /content/cloud-provider-analytics/data/bronze/customers_orgs
users: leídas=800  duplicadas quitadas=0  guardadas=800  -> /content/cloud-provider-analytics/data/bronze/users
billing_monthly: leídas=240  duplicadas quitadas=0  guardadas=240  -> /content/cloud-provider-analytics/data/bronze/billing_monthly
users last_login nulos: 139
billing credits nulos: 137
billing subtotal < 0: 13


In [28]:
for m in ["customers_orgs", "users", "billing_monthly"]:
    print("==", m)
    for ruta, _, archivos in sorted(os.walk(os.path.join(BRONZE, m))):
        for a in sorted(archivos):
            if a.endswith(".parquet"):
                p = os.path.join(ruta, a)
                print(os.path.relpath(p, BRONZE), os.path.getsize(p), "bytes")

== customers_orgs
customers_orgs/plan_tier=enterprise/part-00000-22b0ed0d-e17c-4206-bb05-475080b804cf.c000.snappy.parquet 4437 bytes
customers_orgs/plan_tier=free/part-00000-22b0ed0d-e17c-4206-bb05-475080b804cf.c000.snappy.parquet 4354 bytes
customers_orgs/plan_tier=pro/part-00000-22b0ed0d-e17c-4206-bb05-475080b804cf.c000.snappy.parquet 4868 bytes
customers_orgs/plan_tier=standard/part-00000-22b0ed0d-e17c-4206-bb05-475080b804cf.c000.snappy.parquet 5442 bytes
== users
users/role=admin/part-00000-b6852cb7-92e4-49e0-bdac-912b3c09c3e3.c000.snappy.parquet 5847 bytes
users/role=analyst/part-00000-b6852cb7-92e4-49e0-bdac-912b3c09c3e3.c000.snappy.parquet 6609 bytes
users/role=data_engineer/part-00000-b6852cb7-92e4-49e0-bdac-912b3c09c3e3.c000.snappy.parquet 9007 bytes
users/role=developer/part-00000-b6852cb7-92e4-49e0-bdac-912b3c09c3e3.c000.snappy.parquet 7073 bytes
users/role=devops/part-00000-b6852cb7-92e4-49e0-bdac-912b3c09c3e3.c000.snappy.parquet 7935 bytes
users/role=ml_engineer/part-00000

In [29]:
from pyspark.sql.types import TimestampType, IntegerType, LongType
from pyspark.sql.window import Window

spark.conf.set("spark.sql.session.timeZone", "UTC")

schema_events = StructType([
    StructField("event_id",           StringType(),    True),
    StructField("timestamp",          TimestampType(), True),
    StructField("org_id",             StringType(),    True),
    StructField("resource_id",        StringType(),    True),
    StructField("service",            StringType(),    True),
    StructField("region",             StringType(),    True),
    StructField("metric",             StringType(),    True),
    StructField("value",              StringType(),    True),   # llega como número o como texto
    StructField("unit",               StringType(),    True),
    StructField("cost_usd_increment", DoubleType(),    True),
    StructField("schema_version",     IntegerType(),   True),
    StructField("carbon_kg",          DoubleType(),    True),   # solo en v2
    StructField("genai_tokens",       LongType(),      True),   # solo servicio genai
])

EVENTS_LANDING = os.path.join(LANDING, "usage_events_stream")
REPLAY = os.path.join(REPO, "data/replay/usage_events_stream")

events_all = spark.read.schema(schema_events).json(EVENTS_LANDING)

(
    events_all
    .repartitionByRange(120, "timestamp")      # 120 archivos, cada uno con un tramo de tiempo
    .sortWithinPartitions("timestamp")
    .write.mode("overwrite")
    .json(REPLAY)
)

# Verificación: los archivos tienen que quedar ordenados y sin solaparse
replay = (
    spark.read.schema(schema_events).json(REPLAY)
    .withColumn("f", F.col("_metadata.file_name"))
)
rangos = (
    replay.groupBy("f")
    .agg(F.count("*").alias("n"), F.min("timestamp").alias("desde"), F.max("timestamp").alias("hasta"))
    .orderBy("f")
)
rangos.show(5, truncate=False)

w = Window.orderBy("f")
solapes = (
    rangos.withColumn("desde_sig", F.lead("desde").over(w))
    .filter(F.col("hasta") > F.col("desde_sig"))
    .count()
)
print("Archivos:", rangos.count())
print("Eventos totales:", replay.count())                 # 43200
print("Archivos que se solapan con el siguiente:", solapes)   # 0

+---------------------------------------------------------+---+-------------------+-------------------+
|f                                                        |n  |desde              |hasta              |
+---------------------------------------------------------+---+-------------------+-------------------+
|part-00000-8b88413b-524b-4460-9114-98099ccabbe5-c000.json|361|2025-07-03 00:02:00|2025-07-03 12:53:00|
|part-00001-8b88413b-524b-4460-9114-98099ccabbe5-c000.json|342|2025-07-03 12:54:00|2025-07-03 23:32:00|
|part-00002-8b88413b-524b-4460-9114-98099ccabbe5-c000.json|369|2025-07-03 23:34:00|2025-07-04 12:02:00|
|part-00003-8b88413b-524b-4460-9114-98099ccabbe5-c000.json|340|2025-07-04 12:03:00|2025-07-04 23:02:00|
|part-00004-8b88413b-524b-4460-9114-98099ccabbe5-c000.json|354|2025-07-04 23:03:00|2025-07-05 10:18:00|
+---------------------------------------------------------+---+-------------------+-------------------+
only showing top 5 rows
Archivos: 120
Eventos totales: 43200
Arc

In [30]:
BRONZE_EVENTS = os.path.join(BRONZE, "usage_events_stream")
CHECKPOINT    = os.path.join(REPO, "data/checkpoints/bronze_events")

stream = (
    spark.readStream
    .schema(schema_events)
    .option("maxFilesPerTrigger", 1)          # un archivo por micro-lote
    .json(REPLAY)
)

bronze_stream = (
    stream
    .withColumn("value", F.col("value").cast("double"))        # texto numérico -> número
    .withColumn("ingest_ts", F.current_timestamp())
    .withColumn("source_file", F.col("_metadata.file_name"))
    .withColumn("year", F.year("timestamp"))
    .withColumn("month", F.month("timestamp"))
    .withWatermark("timestamp", "10 minutes")                   # watermark de 10 minutos
    .dropDuplicatesWithinWatermark(["event_id"])                # deduplica por event_id
)

def run_bronze_events():
    q = (
        bronze_stream.writeStream
        .queryName("bronze_events")                             # NUEVO: nombre de la consulta
        .format("parquet")
        .outputMode("append")
        .option("path", BRONZE_EVENTS)
        .option("checkpointLocation", CHECKPOINT)
        .partitionBy("year", "month")                           # una carpeta por mes
        .trigger(availableNow=True)                             # procesa todo lo pendiente y termina
        .start()
    )
    q.awaitTermination()
    return q                                                    # NUEVO: devuelve la consulta

def contar_bronze():
    d = spark.read.parquet(BRONZE_EVENTS)
    print("Eventos en Bronze:", d.count())
    d.groupBy("year", "month").count().orderBy("year", "month").show()

# Primera ejecución
consulta = run_bronze_events()                                  # CAMBIADO: guarda la consulta

p = consulta.lastProgress                                       # NUEVO: monitoreo (diapositiva 22)
print(p["batchId"], p["numInputRows"],
      p["inputRowsPerSecond"], p["processedRowsPerSecond"])

contar_bronze()

241 364 464.8786717752235 493.2249322493225
Eventos en Bronze: 43200
+----+-----+-----+
|year|month|count|
+----+-----+-----+
|2025|    7|20880|
|2025|    8|22320|
+----+-----+-----+



In [31]:
import shutil, glob

def total_bronze():
    return spark.read.parquet(BRONZE_EVENTS).count()

# A) Re-ejecutar sin archivos nuevos: el checkpoint evita reprocesar
antes = total_bronze()
run_bronze_events()
despues = total_bronze()
print(f"A) Re-ejecución sin archivos nuevos -> antes={antes}  después={despues}")

# B) Llega un archivo repetido (mismos event_id): no debe sumar eventos
ultimo = sorted(glob.glob(os.path.join(REPLAY, "part-*.json")))[-1]
repetido = os.path.join(REPLAY, "part-99999-repetido.json")
shutil.copy(ultimo, repetido)

antes = total_bronze()
run_bronze_events()
despues = total_bronze()
print(f"B) Llega un archivo repetido      -> antes={antes}  después={despues}")

os.remove(repetido)   # dejamos el replay como estaba

print("Checkpoint:", os.listdir(CHECKPOINT))

A) Re-ejecución sin archivos nuevos -> antes=43200  después=43200
B) Llega un archivo repetido      -> antes=43200  después=43200
Checkpoint: ['state', '.metadata.crc', 'offsets', 'metadata', 'sources', 'commits']


In [32]:
SILVER_EVENTS     = os.path.join(REPO, "data/silver/usage_events")
QUARANTINE_EVENTS = os.path.join(REPO, "data/quarantine/usage_events")
CHECKPOINT_SILVER = os.path.join(REPO, "data/checkpoints/silver_events")

customers_dim = (
    spark.read.parquet(os.path.join(BRONZE, "customers_orgs"))
    .select("org_id", "industry", F.col("hq_region").alias("org_hq_region"),
            "plan_tier", "is_enterprise")
)

unidad_por_metrica = (
    F.when(F.col("metric") == "cpu_hours", "hours")
     .when(F.col("metric") == "requests", "count")
     .when(F.col("metric") == "storage_gb_hours", "gb_hours")
)

def procesar_lote(batch_df, batch_id):
    motivo = (
        F.when(F.col("event_id").isNull(), "event_id_nulo")
         .when(F.col("cost_usd_increment") < -0.01, "cost_usd_increment_menor_a_-0.01")
    )
    marcado = batch_df.withColumn("quarantine_reason", motivo)

    invalidos = (
        marcado.filter(F.col("quarantine_reason").isNotNull())
        .withColumn("quarantine_ts", F.current_timestamp())
    )
    validos = marcado.filter(F.col("quarantine_reason").isNull()).drop("quarantine_reason")

    silver = (
        validos
        .withColumn("unit_imputed", F.col("unit").isNull())
        .withColumn("unit", F.coalesce(F.col("unit"), unidad_por_metrica))
        .withColumn("value_missing", F.col("value").isNull())
        .withColumn("usage_date", F.to_date("timestamp"))
        .withColumn("requests",         F.when(F.col("metric") == "requests",         F.col("value")))
        .withColumn("cpu_hours",        F.when(F.col("metric") == "cpu_hours",        F.col("value")))
        .withColumn("storage_gb_hours", F.when(F.col("metric") == "storage_gb_hours", F.col("value")))
        .join(F.broadcast(customers_dim), on="org_id", how="left")
    )

    silver.write.mode("append").partitionBy("year", "month").parquet(SILVER_EVENTS)
    invalidos.write.mode("append").parquet(QUARANTINE_EVENTS)

bronze_schema = spark.read.parquet(BRONZE_EVENTS).schema
silver_stream = (
    spark.readStream.schema(bronze_schema)
    .option("maxFilesPerTrigger", 10)
    .parquet(BRONZE_EVENTS)
)

def run_silver_events():
    q = (
        silver_stream.writeStream
        .foreachBatch(procesar_lote)
        .option("checkpointLocation", CHECKPOINT_SILVER)
        .trigger(availableNow=True)
        .start()
    )
    q.awaitTermination()

run_silver_events()

# Verificación
s = spark.read.parquet(SILVER_EVENTS)
q = spark.read.parquet(QUARANTINE_EVENTS)
n_s, n_q = s.count(), q.count()
print("Silver:", n_s)                                              # 42989
print("Quarantine:", n_q)                                          # 211
print("% inválidos:", round(n_q / (n_s + n_q) * 100, 2))           # 0.49
s.groupBy("year", "month").count().orderBy("year", "month").show()  # 20778 / 22211
print("unit nulos en Silver:", s.filter(F.col("unit").isNull()).count())          # 0
print("unit imputada:", s.filter("unit_imputed").count())                         # 2069
print("value nulo:", s.filter("value_missing").count())                           # 873
q.groupBy("quarantine_reason").count().show(truncate=False)
q.select("event_id", "cost_usd_increment", "quarantine_reason", "source_file") \
 .orderBy("cost_usd_increment").show(5, truncate=False)                           # mínimo -154.46
s.select("event_id", "service", "metric", "unit", "requests", "cpu_hours",
         "storage_gb_hours", "industry", "plan_tier").show(5, truncate=False)

Silver: 42989
Quarantine: 211
% inválidos: 0.49
+----+-----+-----+
|year|month|count|
+----+-----+-----+
|2025|    7|20778|
|2025|    8|22211|
+----+-----+-----+

unit nulos en Silver: 0
unit imputada: 2069
value nulo: 873
+--------------------------------+-----+
|quarantine_reason               |count|
+--------------------------------+-----+
|cost_usd_increment_menor_a_-0.01|211  |
+--------------------------------+-----+

+----------------+------------------+--------------------------------+---------------------------------------------------------+
|event_id        |cost_usd_increment|quarantine_reason               |source_file                                              |
+----------------+------------------+--------------------------------+---------------------------------------------------------+
|evt_t5brta67j5l3|-154.4608         |cost_usd_increment_menor_a_-0.01|part-00105-5fb1b7d7-9c03-47d4-8ea0-d2e2bcb64c11-c000.json|
|evt_9ur456xoexl5|-19.8245          |cost_usd_increment

In [33]:
# Silver de billing_monthly: créditos, moneda normalizada a USD y regla de calidad
SILVER_BILLING     = os.path.join(REPO, "data/silver/billing_monthly")
QUARANTINE_BILLING = os.path.join(REPO, "data/quarantine/billing_monthly")

billing_bronze = spark.read.parquet(os.path.join(BRONZE, "billing_monthly"))

tasa_valida = F.coalesce(F.col("exchange_rate_to_usd") > 0, F.lit(False))

billing_marcado = (
    billing_bronze
    .withColumn("credits_imputed", F.col("credits").isNull())           # marca cuáles se completaron
    .withColumn("credits", F.coalesce(F.col("credits"), F.lit(0.0)))    # credits nulo = 0
    .withColumn("is_credit", F.col("subtotal") < 0)                     # las 13 facturas negativas
    .withColumn("net_usd", F.round(
        (F.col("subtotal") - F.col("credits")) * F.col("exchange_rate_to_usd"), 2))
    .withColumn("quarantine_reason",
        F.when(~tasa_valida, "exchange_rate_nula_o_no_positiva"))
)

billing_validos   = billing_marcado.filter(F.col("quarantine_reason").isNull()).drop("quarantine_reason")
billing_invalidos = (billing_marcado.filter(F.col("quarantine_reason").isNotNull())
                     .withColumn("quarantine_ts", F.current_timestamp()))

# overwrite: re-ejecutable, pisa el resultado anterior
billing_validos.coalesce(1).write.mode("overwrite").partitionBy("month").parquet(SILVER_BILLING)
billing_invalidos.coalesce(1).write.mode("overwrite").parquet(QUARANTINE_BILLING)

# Verificación
s = spark.read.parquet(SILVER_BILLING)
print("Filas en Silver:", s.count())                                        # 240
print("Quarantine:", billing_invalidos.count())                             # 0
print("credits imputados a 0:", s.filter("credits_imputed").count())        # 137
print("Facturas con crédito (subtotal < 0):", s.filter("is_credit").count())  # 13
s.groupBy("currency").count().orderBy("currency").show()                    # ARS 51 · EUR 29 · USD 160
total_org = s.filter(F.col("org_id") == "org_xaji0y6d").agg(F.sum("net_usd")).first()[0]
print("Total org_xaji0y6d (USD):", round(total_org, 2))                     # 1729.18, igual que la sección 9

Filas en Silver: 240
Quarantine: 0
credits imputados a 0: 137
Facturas con crédito (subtotal < 0): 13
+--------+-----+
|currency|count|
+--------+-----+
|     ARS|   51|
|     EUR|   29|
|     USD|  160|
+--------+-----+

Total org_xaji0y6d (USD): 1729.18


In [34]:
GOLD_USAGE = os.path.join(REPO, "data/gold/org_daily_usage_by_service")

silver = spark.read.parquet(SILVER_EVENTS)

org_daily_usage_by_service = (
    silver
    .groupBy("org_id", "usage_date", "service")
    .agg(
        F.round(F.sum("cost_usd_increment"), 4).alias("daily_cost_usd"),
        F.sum("requests").alias("requests"),
        F.sum("cpu_hours").alias("cpu_hours"),
        F.sum("storage_gb_hours").alias("storage_gb_hours"),
        F.sum("genai_tokens").alias("genai_tokens"),
        F.sum("carbon_kg").alias("carbon_kg"),
        F.count("*").alias("n_events"),
    )
)

# Evidencia del shuffle (captura para evidence/)
org_daily_usage_by_service.explain(mode="formatted")

(
    org_daily_usage_by_service
    .coalesce(1)
    .write.mode("overwrite")
    .parquet(GOLD_USAGE)
)

# Verificación
gold = spark.read.parquet(GOLD_USAGE)
print("Filas Gold:", gold.count())                                        # 11050
claves = gold.select("org_id", "usage_date", "service").distinct().count()
print("Claves únicas (grano):", claves)                                   # 11050
print("Suma n_events:", gold.agg(F.sum("n_events")).first()[0])           # 42989
print("Costo total:", round(gold.agg(F.sum("daily_cost_usd")).first()[0], 2))   # ~148351.58
print("Requests total:", gold.agg(F.sum("requests")).first()[0])          # 2281020.0
print("genai_tokens total:", gold.agg(F.sum("genai_tokens")).first()[0])  # 2545481
print("Filas con genai_tokens:", gold.filter(F.col("genai_tokens").isNotNull()).count())  # 848
gold.orderBy("org_id", "usage_date", "service").show(5, truncate=False)

== Physical Plan ==
AdaptiveSparkPlan (6)
+- HashAggregate (5)
   +- Exchange (4)
      +- HashAggregate (3)
         +- Project (2)
            +- Scan parquet  (1)


(1) Scan parquet 
Output [11]: [org_id#26797, service#26801, cost_usd_increment#26806, carbon_kg#26808, genai_tokens#26809L, usage_date#26814, requests#26815, cpu_hours#26816, storage_gb_hours#26817, year#26822, month#26823]
Batched: true
Location: InMemoryFileIndex [file:/content/cloud-provider-analytics/data/silver/usage_events]
ReadSchema: struct<org_id:string,service:string,cost_usd_increment:double,carbon_kg:double,genai_tokens:bigint,usage_date:date,requests:double,cpu_hours:double,storage_gb_hours:double>

(2) Project
Output [9]: [org_id#26797, service#26801, cost_usd_increment#26806, carbon_kg#26808, genai_tokens#26809L, usage_date#26814, requests#26815, cpu_hours#26816, storage_gb_hours#26817]
Input [11]: [org_id#26797, service#26801, cost_usd_increment#26806, carbon_kg#26808, genai_tokens#26809L, usage_date#268

In [35]:
def tam(ruta):
    total, n = 0, 0
    for carpeta, _, archivos in os.walk(ruta):
        for a in archivos:
            if a.endswith((".parquet", ".json", ".jsonl")):
                total += os.path.getsize(os.path.join(carpeta, a))
                n += 1
    return n, round(total / 1024 / 1024, 2)

zonas = {
    "Landing (eventos)": EVENTS_LANDING,
    "Bronze": BRONZE,
    "Silver": SILVER_EVENTS,
    "Gold": GOLD_USAGE,
    "Quarantine": QUARANTINE_EVENTS,
}

for nombre, ruta in zonas.items():
    n, mb = tam(ruta)
    print(f"{nombre:20s} {n:4d} archivos  {mb:8.2f} MB  {ruta}")

Landing (eventos)     120 archivos     12.33 MB  /content/cloud-provider-analytics/data/extracted/datalake/landing/usage_events_stream
Bronze                981 archivos      7.55 MB  /content/cloud-provider-analytics/data/bronze
Silver                196 archivos      4.10 MB  /content/cloud-provider-analytics/data/silver/usage_events
Gold                    1 archivos      0.26 MB  /content/cloud-provider-analytics/data/gold/org_daily_usage_by_service
Quarantine            157 archivos      0.84 MB  /content/cloud-provider-analytics/data/quarantine/usage_events


In [36]:
TEST_LATE = os.path.join(REPO, "data/test_late")
shutil.rmtree(TEST_LATE, ignore_errors=True)       # empezar limpio

late_stream = (
    spark.readStream
    .schema(schema_events)
    .option("maxFilesPerTrigger", 1)               # un archivo por micro-lote
    .json(EVENTS_LANDING)                          # archivos ORIGINALES, sin ordenar
    .withWatermark("timestamp", "10 minutes")
    .dropDuplicatesWithinWatermark(["event_id"])
)

q = (
    late_stream.writeStream
    .format("parquet")
    .outputMode("append")
    .option("path", os.path.join(TEST_LATE, "out"))
    .option("checkpointLocation", os.path.join(TEST_LATE, "checkpoint"))
    .trigger(availableNow=True)
    .start()
)
q.awaitTermination()

total = 43200
pasaron = spark.read.parquet(os.path.join(TEST_LATE, "out")).count()
descartados = total - pasaron
print("Eventos de entrada:", total)
print("Pasaron el watermark:", pasaron)                                  # ~731
print("Descartados por llegar tarde:", descartados)                      # ~42469
print("% descartado:", round(descartados / total * 100, 2))              # ~98.3

# Contraste: el replay ordenado (Bronze) no pierde nada
print("Bronze con replay ordenado:", spark.read.parquet(BRONZE_EVENTS).count())   # 43200

# Idempotencia de Silver: re-ejecutar no cambia el conteo
antes = spark.read.parquet(SILVER_EVENTS).count()
run_silver_events()
despues = spark.read.parquet(SILVER_EVENTS).count()
print(f"Silver re-ejecutado -> antes={antes}  después={despues}")        # 42989 / 42989

Eventos de entrada: 43200
Pasaron el watermark: 726
Descartados por llegar tarde: 42474
% descartado: 98.32
Bronze con replay ordenado: 43200
Silver re-ejecutado -> antes=42989  después=42989


In [42]:
# Prueba controlada de late data: un evento que llega después del watermark se descarta y se cuenta
import json

TEST_TARDIO = os.path.join(REPO, "data/test_tardio")
shutil.rmtree(TEST_TARDIO, ignore_errors=True)
os.makedirs(os.path.join(TEST_TARDIO, "in"))

def evento(event_id, ts):
    return {"event_id": event_id, "timestamp": ts, "org_id": "org_test",
            "service": "compute", "metric": "requests", "value": "1", "unit": "count",
            "cost_usd_increment": 0.1, "schema_version": 2}

def escribir(nombre, eventos):
    with open(os.path.join(TEST_TARDIO, "in", nombre), "w") as f:
        for e in eventos:
            f.write(json.dumps(e) + "\n")

escribir("lote_1.json", [evento("t1", "2025-08-01T10:00:00Z"),
                         evento("t2", "2025-08-01T10:05:00Z")])
escribir("lote_2.json", [evento("t3", "2025-08-01T10:30:00Z")])   # watermark pasa a 10:20
escribir("lote_3.json", [evento("t4", "2025-08-01T10:02:00Z")])   # muy tardío: 10:02 es anterior a 10:20

tardio_stream = (
    spark.readStream
    .schema(schema_events)
    .option("maxFilesPerTrigger", 1)                  # un archivo por micro-lote, como en Bronze
    .json(os.path.join(TEST_TARDIO, "in"))
    .withWatermark("timestamp", "10 minutes")
    .dropDuplicatesWithinWatermark(["event_id"])
)

q = (
    tardio_stream.writeStream
    .format("parquet")
    .outputMode("append")
    .option("path", os.path.join(TEST_TARDIO, "out"))
    .option("checkpointLocation", os.path.join(TEST_TARDIO, "checkpoint"))
    .trigger(availableNow=True)
    .start()
)
q.awaitTermination()

salida = spark.read.parquet(os.path.join(TEST_TARDIO, "out"))
print("Eventos enviados: 4")
print("Eventos que pasaron:", salida.count())                     # 3
salida.select("event_id", "timestamp").orderBy("timestamp").show(truncate=False)   # t1, t2, t3

descartados = sum(
    p["stateOperators"][0]["numRowsDroppedByWatermark"]
    for p in q.recentProgress if p["stateOperators"]
)
print("Descartados por watermark (métrica de Spark):", descartados)   # 1

Eventos enviados: 4
Eventos que pasaron: 3
+--------+-------------------+
|event_id|timestamp          |
+--------+-------------------+
|t1      |2025-08-01 10:00:00|
|t2      |2025-08-01 10:05:00|
|t3      |2025-08-01 10:30:00|
+--------+-------------------+

Descartados por watermark (métrica de Spark): 1


In [37]:
# prueba de conexión a Astra (Cassandra)
!pip -q install cassandra-driver

import glob
from google.colab import userdata
from cassandra.cluster import Cluster
from cassandra.auth import PlainTextAuthProvider

bundle = glob.glob("/content/secure-connect*.zip")[0]
# Removemos posibles comillas dobles que tenga el token guardado en Secrets
token = userdata.get("ASTRA_TOKEN").strip('"')

cluster = Cluster(
    cloud={"secure_connect_bundle": bundle},
    auth_provider=PlainTextAuthProvider("token", token),
)
session = cluster.connect("cloud_provider")

fila = session.execute("SELECT release_version FROM system.local").one()
print("Conectado. Versión de Cassandra:", fila.release_version)
print("Keyspace en uso:", session.keyspace)

Conectado. Versión de Cassandra: 4.0.11.0-6819aa9c7f6c
Keyspace en uso: cloud_provider


In [38]:
# tabla 1 en Cassandra + carga desde Gold
# Clave: partición por org_id y orden por (usage_date, service), porque las tres
# áreas consultan por organización y período (ver evidence/02_decisiones/decisiones.md).
from cassandra.concurrent import execute_concurrent_with_args

GOLD_PATH = os.path.join(REPO, "data/gold/org_daily_usage_by_service")

# Se recrea la tabla en cada corrida: así el resultado es el mismo desde cero.
session.execute("DROP TABLE IF EXISTS cloud_provider.org_daily_usage_by_service")
session.execute("""
CREATE TABLE cloud_provider.org_daily_usage_by_service (
    org_id text,
    usage_date date,
    service text,
    daily_cost_usd double,
    requests bigint,
    cpu_hours double,
    storage_gb_hours double,
    genai_tokens bigint,
    carbon_kg double,
    n_events bigint,
    PRIMARY KEY ((org_id), usage_date, service)
) WITH CLUSTERING ORDER BY (usage_date ASC, service ASC)
""")

filas = spark.read.parquet(GOLD_PATH).collect()

insert = session.prepare("""
INSERT INTO cloud_provider.org_daily_usage_by_service
(org_id, usage_date, service, daily_cost_usd, requests, cpu_hours,
 storage_gb_hours, genai_tokens, carbon_kg, n_events)
VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
""")

# requests es double en Gold pero todos sus valores son enteros (verificado:
# 0 filas con decimales y suma igual en Gold y Cassandra), por eso va como bigint.
args = [(r["org_id"], r["usage_date"], r["service"], r["daily_cost_usd"],
         None if r["requests"] is None else int(r["requests"]),
         r["cpu_hours"], r["storage_gb_hours"], r["genai_tokens"],
         r["carbon_kg"], r["n_events"]) for r in filas]

resultados = execute_concurrent_with_args(session, insert, args, concurrency=50)
errores = [r for ok, r in resultados if not ok]

en_cassandra = session.execute(
    "SELECT COUNT(*) FROM cloud_provider.org_daily_usage_by_service").one()[0]
print("Filas en Gold:", len(filas))
print("Filas en Cassandra:", en_cassandra)
print("Errores de escritura:", len(errores))

Filas en Gold: 11050
Filas en Cassandra: 11050
Errores de escritura: 0


In [39]:
from pyspark.sql import functions as F

g = spark.read.parquet(GOLD_PATH)
print("Gold: filas con requests no entero:",
      g.filter(F.col("requests") != F.floor("requests")).count())
print("Gold: suma de requests:", g.agg(F.sum("requests")).first()[0])

c = session.execute("SELECT SUM(requests) FROM cloud_provider.org_daily_usage_by_service").one()[0]
print("Cassandra: suma de requests:", c)

Gold: filas con requests no entero: 0
Gold: suma de requests: 2281020.0


Cassandra: suma de requests: 2281020


In [40]:
# consulta CQL 1 — costos y requests diarios de una organización en un rango
q1 = """
SELECT usage_date, service, daily_cost_usd, requests
FROM cloud_provider.org_daily_usage_by_service
WHERE org_id = 'org_dbdw2pcn'
  AND usage_date >= '2025-07-01'
  AND usage_date <= '2025-07-14'
"""
r1 = list(session.execute(q1))
for f in r1:
    print(f.usage_date, f.service, f.daily_cost_usd, f.requests)
print("Filas:", len(r1), "| costo total:", round(sum(f.daily_cost_usd for f in r1), 4))

2025-07-03 database 20.463 383
2025-07-04 database 9.1282 120
2025-07-05 compute 10.5084 110
2025-07-05 database 20.5993 367
2025-07-05 networking 1.806 125
2025-07-06 compute 26.692 299
2025-07-06 database 15.5938 328
2025-07-06 networking 0.3233 None
2025-07-07 compute 18.8965 267
2025-07-07 database 1.7029 None
2025-07-07 networking 2.6985 240
2025-07-08 compute 20.5009 251
2025-07-08 database 6.8366 110
2025-07-08 networking 1.3052 116
2025-07-09 compute 20.7514 257
2025-07-09 database 10.7993 232
2025-07-09 networking 1.5492 135
2025-07-10 compute 21.7875 277
2025-07-10 database 11.9865 216
2025-07-10 networking 0.2637 None
2025-07-11 compute 19.1478 226
2025-07-11 database 17.7013 357
2025-07-11 networking 0.9433 102
2025-07-12 compute 1.6501 None
2025-07-12 database 16.9153 352
2025-07-12 networking 1.6244 205
2025-07-13 compute 39.8767 459
2025-07-13 database 8.9409 134
2025-07-13 networking 0.0663 None
2025-07-14 compute 21.7431 245
2025-07-14 database 8.6132 146
Filas: 31 | c

In [41]:
# consulta CQL 2 — servicios que más costaron en los últimos 14 días de una organización
from collections import defaultdict

q2 = """
SELECT usage_date, service, daily_cost_usd
FROM cloud_provider.org_daily_usage_by_service
WHERE org_id = 'org_dbdw2pcn'
  AND usage_date >= '2025-08-18'
  AND usage_date <= '2025-08-31'
"""
r2 = list(session.execute(q2))

# Cassandra devuelve las filas de la partición de la organización en ese rango;
# la suma por servicio y el orden se hacen acá, porque CQL no agrupa por una
# columna que está después de usage_date en la clave.
costo_por_servicio = defaultdict(float)
for f in r2:
    costo_por_servicio[f.service] += f.daily_cost_usd

print("Filas leídas de Cassandra:", len(r2))
for servicio, costo in sorted(costo_por_servicio.items(), key=lambda x: -x[1]):
    print(servicio, round(costo, 2))

Filas leídas de Cassandra: 31
compute 278.1
database 133.29
networking 8.39
